In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv(r"E:\AI-Customer-Support-Chatbot\data\processed\processed_data.csv")

In [ ]:
# Samples per category
category_counts = df['category'].value_counts()
category_percentage = (category_counts / len(df)) * 100

plt.figure(figsize=(10, 5))
plt.bar(category_counts.index, category_counts.values)
plt.title('Samples per Category')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 intents
intent_counts = df['intent'].value_counts().head(15)
top15 = intent_counts.head(15)
percentages = (top15 / len(df)) * 100

for i, value in enumerate(top15.values[::-1]):
    pct = percentages.iloc[::-1].iloc[i]
    plt.text(i, value, f'{value}\n({pct:.1f}%)', ha='center')

plt.figure(figsize=(10, 5))
plt.bar(intent_counts.index, intent_counts.values)
plt.title('Top 15 Intents')
plt.xlabel('Intent')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Token count distribution
plt.figure(figsize=(9, 5))
plt.hist(df['token_count'], bins=15)

mean_val = df['token_count'].mean()
median_val = df['token_count'].median()

plt.text(0.7, 0.9, f"Mean = {mean_val:.1f}", transform=plt.gca().transAxes)
plt.text(0.7, 0.85, f"Median = {median_val:.1f}", transform=plt.gca().transAxes)

plt.title('Token Count Distribution')
plt.xlabel('Number of Tokens')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Response word count distribution
if 'response_word_count' not in df.columns:
    df['response_word_count'] = df['response'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(9, 5))
plt.hist(df['response_word_count'], bins=20)
plt.title('Response Word Count Distribution')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

print(f"Mean: {df['response_word_count'].mean():.0f} words")

In [ ]:
# Placeholder distribution
if 'placeholder_count' not in df.columns:
    df['placeholder_count'] = df['instruction_clean'].apply(
        lambda x: str(x).count('<PLACEHOLDER>')
    )

ph_counts = df['placeholder_count'].value_counts().sort_index()
labels = [f"{v} placeholder" if v == 1 else f"{v} placeholders" for v in ph_counts.index]

plt.figure(figsize=(6, 6))
plt.pie(ph_counts.values, labels=labels, autopct='%1.1f%%')
plt.title('Placeholder Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 content words
STOPWORDS = {
    'i', 'a', 'the', 'to', 'my', 'and', 'of', 'for', 'in', 'is', 'it',
    'me', 'on', 'you', 'your', 'this', 'that', 'with', 'about', 'have',
    'want', 'can', 'do', 'not', 'an', 'be', 'at', 'or', 'has', 'am',
    'are', 'was', 'get', 'we', 'so', 'how', 'need', 'would', 'please',
    'could', 'help', 's'
}

all_words = [w for tokens in df['tokens'] for w in tokens if w not in STOPWORDS]
word_freq = pd.Series(all_words).value_counts().head(20)

plt.figure(figsize=(11, 5))
plt.bar(word_freq.index, word_freq.values)
plt.title('Top 20 Content Words in Instructions')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Token count by category (boxplot)
categories = df['category'].unique()
grouped = [df[df['category'] == cat]['token_count'].values for cat in categories]

plt.figure(figsize=(12, 5))
plt.boxplot(grouped, labels=categories)
plt.title('Token Count Distribution by Category')
plt.xlabel('Category')
plt.ylabel('Token Count')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Mean response word count by category
resp_stats = df.groupby('category')['response_word_count'].mean().sort_values(ascending=False)

plt.figure(figsize=(11, 5))
plt.bar(resp_stats.index, resp_stats.values)
plt.title('Mean Response Word Count by Category')
plt.xlabel('Category')
plt.ylabel('Mean Word Count')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Instruction token count vs response word count (scatter)
plt.figure(figsize=(9, 6))
for cat in df['category'].unique():
    sub = df[df['category'] == cat]
    plt.scatter(sub['token_count'], sub['response_word_count'], label=cat, alpha=0.4, s=15)

plt.title('Instruction Token Count vs Response Word Count')
plt.xlabel('Instruction Token Count')
plt.ylabel('Response Word Count')
plt.legend(fontsize=8)
plt.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print(df[['token_count', 'response_word_count', 'placeholder_count']].corr().round(3))

In [ ]:
# Mean placeholder count by category
ph_rate = df.groupby('category')['placeholder_count'].mean().sort_values(ascending=False)

plt.figure(figsize=(11, 5))
plt.bar(ph_rate.index, ph_rate.values)
plt.title('Mean Placeholder Count per Instruction by Category')
plt.xlabel('Category')
plt.ylabel('Mean Placeholder Count')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Unique intents per category
intent_diversity = df.groupby('category')['intent'].nunique().sort_values(ascending=False)

plt.figure(figsize=(11, 5))
plt.bar(intent_diversity.index, intent_diversity.values)
plt.title('Number of Unique Intents per Category')
plt.xlabel('Category')
plt.ylabel('Unique Intent Count')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
#mean token count by category x top 10 intents
import math

top_intents = df['intent'].value_counts().head(10).index.tolist()
sub = df[df['intent'].isin(top_intents)]
pivot = sub.pivot_table(values='token_count', index='category', columns='intent', aggfunc='mean')

plt.figure(figsize=(13, 6))
plt.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=35, ha='right', fontsize=9)
plt.yticks(range(len(pivot.index)), pivot.index, fontsize=9)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not math.isnan(val):
            plt.text(j, i, f"{val:.1f}", ha='center', va='center', fontsize=8)

plt.colorbar(label='Mean Token Count')
plt.title('Mean Instruction Token Count: Category x Top 10 Intents')
plt.tight_layout()
plt.show()